# 06 — Model Training (Clean Version)
### Student Performance Analysis System (SPAS)

This notebook trains the ML models required by the SPAS system.

### Design Decisions
- **Grade letter classifier** — skipped. `final_score` was IQR-capped to 43.85–87.98 during cleaning, eliminating grades A and most of B. Grade letter is derived from `predicted_score` at inference time in `predictor.py`.
- **Pass/Fail classifier** — skipped. XGBoost classifier failed to learn the binary boundary reliably (Fail F1 = 0.30) because the 60-point threshold falls in a dense compressed region of the capped score range. Pass/fail and `failure_probability` are derived from `predicted_score` at inference time.
- **Final models saved:** `grade_predictor.pkl`, `feature_scaler.pkl`, `feature_list.pkl`

**Input:**  `data/features/dataset_v1_selected.csv` + `models/feature_list.pkl`  
**Output:** `models/grade_predictor.pkl`, `models/feature_scaler.pkl`

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

os.makedirs('../models', exist_ok=True)
os.makedirs('../eda',    exist_ok=True)

RANDOM_SEED = 42
print('Imports done.')

Imports done.


## 2. Load Dataset and Feature List

In [3]:
df = pd.read_csv('../data/features/features_selectionv2.csv')
print(f'Dataset shape : {df.shape}')
print(f'Columns       : {df.columns.tolist()}')

selected_features = joblib.load('../models/feature_list.pkl')
print(f'\nFeature list loaded ({len(selected_features)} features):')
for i, f in enumerate(selected_features, 1):
    print(f'  {i}. {f}')

Dataset shape : (5000, 9)
Columns       : ['attendance_percentage', 'midterm_score', 'historical_gpa', 'study_hours_per_week', 'subject_difficulty_score', 'ca_avg', 'rule_risk_score', 'final_score', 'pass_fail']

Feature list loaded (7 features):
  1. attendance_percentage
  2. midterm_score
  3. historical_gpa
  4. study_hours_per_week
  5. subject_difficulty_score
  6. ca_avg
  7. rule_risk_score


## 3. Prepare Features and Target

In [4]:
X     = df[selected_features]
y_reg = df['final_score']

print('── Regression Target (final_score) ──')
print(f'  Mean : {y_reg.mean():.2f}')
print(f'  Std  : {y_reg.std():.2f}')
print(f'  Min  : {y_reg.min():.2f}')
print(f'  Max  : {y_reg.max():.2f}')
print(f'  Shape: {X.shape}')

── Regression Target (final_score) ──
  Mean : 66.02
  Std  : 8.18
  Min  : 43.85
  Max  : 87.98
  Shape: (5000, 7)


## 4. Train / Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg,
    test_size=0.2,
    random_state=RANDOM_SEED
)

print('── Train / Test Split ──')
print(f'  Training rows : {len(X_train):,}  (80%)')
print(f'  Test rows     : {len(X_test):,}   (20%)')

── Train / Test Split ──
  Training rows : 4,000  (80%)
  Test rows     : 1,000   (20%)


## 5. Feature Scaling

In [6]:
# Fit ONLY on training data — never on test data
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save immediately
joblib.dump(scaler, '../models/feature_scaler.pkl')

print('── Feature Scaler ──')
print(f'  Fitted on : {X_train_scaled.shape[0]:,} training rows')
print(f'  Features  : {X_train_scaled.shape[1]}')
print(f'  Saved to  : models/feature_scaler.pkl')
print()

scaler_stats = pd.DataFrame({
    'feature': selected_features,
    'mean'   : scaler.mean_.round(4),
    'std'    : scaler.scale_.round(4)
})
print(scaler_stats.to_string(index=False))

── Feature Scaler ──
  Fitted on : 4,000 training rows
  Features  : 7
  Saved to  : models/feature_scaler.pkl

                 feature    mean     std
   attendance_percentage 82.6746 10.0941
           midterm_score 64.2078 11.4020
          historical_gpa  2.9570  0.4911
    study_hours_per_week 22.5177  4.4881
subject_difficulty_score  0.5559  0.2603
                  ca_avg 64.6440  9.9813
         rule_risk_score  0.2827  0.0777


## 6. Model Comparison — 5-Fold Cross Validation
Compare three regression models before committing to XGBoost.

In [7]:
print('── Regression Model Comparison (5-Fold CV) ──\n')

regression_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest'    : RandomForestRegressor(
                            n_estimators=200, max_depth=8,
                            random_state=RANDOM_SEED, n_jobs=-1),
    'XGBoost'          : XGBRegressor(
                            n_estimators=200, learning_rate=0.1,
                            max_depth=6, random_state=RANDOM_SEED,
                            verbosity=0)
}

cv_results = {}
for name, model in regression_models.items():
    cv_scores = cross_val_score(
        model, X_train_scaled, y_train,
        cv=5, scoring='r2', n_jobs=-1
    )
    cv_results[name] = cv_scores
    status = '✅' if cv_scores.mean() >= 0.80 else '⚠️ '
    print(f'  {status} {name:<22}: R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\n  Target  : R² ≥ 0.80')
print('  Selected: XGBoost Regressor (per Section 8.7 of SPAS doc)')

── Regression Model Comparison (5-Fold CV) ──

  ✅ Linear Regression     : R² = 0.8611 ± 0.0038
  ✅ Random Forest         : R² = 0.8494 ± 0.0054
  ✅ XGBoost               : R² = 0.8379 ± 0.0071

  Target  : R² ≥ 0.80
  Selected: XGBoost Regressor (per Section 8.7 of SPAS doc)


## 7. Train Final XGBoost Regressor

In [8]:
print('Training final XGBoost Regressor on full training set...')

xgb_regressor = XGBRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
    verbosity=0
)

xgb_regressor.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False
)

# Evaluate on test set
y_pred = xgb_regressor.predict(X_test_scaled)
y_pred = np.clip(y_pred, 0, 100)   # clamp to valid score range

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('\n── XGBoost Regressor — Test Set Results ──')
print(f'  MAE  : {mae:.4f}   {"✅ OK" if mae  < 8    else "⚠️  Above target"} (target < 8)')
print(f'  RMSE : {rmse:.4f}  {"✅ OK" if rmse < 12   else "⚠️  Above target"} (target < 12)')
print(f'  R²   : {r2:.4f}   {"✅ OK" if r2   >= 0.80 else "⚠️  Below target"} (target ≥ 0.80)')

# Save model
joblib.dump(xgb_regressor, '../models/grade_predictor.pkl')
print('\n  Saved → models/grade_predictor.pkl')

Training final XGBoost Regressor on full training set...

── XGBoost Regressor — Test Set Results ──
  MAE  : 2.5951   ✅ OK (target < 8)
  RMSE : 3.2475  ✅ OK (target < 12)
  R²   : 0.8437   ✅ OK (target ≥ 0.80)

  Saved → models/grade_predictor.pkl


## 8. Model Comparison Bar Chart

In [9]:
model_names = list(cv_results.keys())
means  = [cv_results[m].mean() for m in model_names]
stds   = [cv_results[m].std()  for m in model_names]
colors = ['steelblue', 'seagreen', 'darkorange']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    model_names, means, yerr=stds,
    capsize=6, color=colors,
    edgecolor='white', linewidth=0.8, alpha=0.85
)
ax.axhline(0.80, color='red', linestyle='--',
           linewidth=1.5, label='Target R² = 0.80')
ax.set_ylabel('Cross-Validated R² Score')
ax.set_title('Regression Model Comparison — 5-Fold CV R²', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend()

for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{mean:.4f}', ha='center', va='bottom',
        fontsize=10, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('../eda/model_comparison_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved → eda/model_comparison_regression.png')

Plot saved → eda/model_comparison_regression.png


## 9. Verify PKL Files

In [10]:
pkl_files = {
    'grade_predictor.pkl' : '../models/grade_predictor.pkl',
    'feature_scaler.pkl'  : '../models/feature_scaler.pkl',
    'feature_list.pkl'    : '../models/feature_list.pkl',
}

print('── PKL File Verification ──')
all_ok = True
for name, path in pkl_files.items():
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if exists else 0
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {name:<35} {size:>8.1f} KB')
    if not exists:
        all_ok = False

print()
print('  ✅ All 3 PKL files ready.' if all_ok else '  ❌ Some files missing.')

── PKL File Verification ──
  ✅  grade_predictor.pkl                    811.5 KB
  ✅  feature_scaler.pkl                       1.1 KB
  ✅  feature_list.pkl                         0.1 KB

  ✅ All 3 PKL files ready.


## 10. Sanity Check — Run Live Predictions
Simulates exactly what FastAPI will do at inference time.

In [14]:
# Load everything fresh from disk
loaded_regressor = joblib.load('../models/grade_predictor.pkl')
loaded_scaler    = joblib.load('../models/feature_scaler.pkl')
loaded_features  = joblib.load('../models/feature_list.pkl')

# ── Grade and risk derivation (replaces separate classifiers) ─────────────────
def derive_grade(score: float) -> str:
    """
    Map predicted score to letter grade.
    Custom grading scale for SPAS system.
    """
    if score >= 90:   return 'A+'
    elif score >= 80: return 'A'
    elif score >= 70: return 'B+'
    elif score >= 60: return 'B'
    elif score >= 50: return 'C+'
    elif score >= 40: return 'C'
    elif score >= 30: return 'D+'
    elif score >= 20: return 'D'
    else:             return 'E'

def derive_failure_probability(score: float) -> float:
    """
    Soft failure probability derived from distance to 60-point boundary.
    Score = 60  → prob = 0.50  (right on the boundary)
    Score = 40  → prob = 1.00  (very likely to fail)
    Score = 80  → prob = 0.00  (very unlikely to fail)
    """
    prob = (60 - score) / 20 + 0.5
    return round(float(np.clip(prob, 0.0, 1.0)), 4)

def derive_risk_level(failure_prob: float) -> str:
    """Map failure probability to risk level — Section 8.10 of SPAS doc."""
    if failure_prob >= 0.75:   return 'CRITICAL'
    elif failure_prob >= 0.55: return 'HIGH'
    elif failure_prob >= 0.35: return 'MEDIUM'
    else:                      return 'LOW'

def predict_student(features: dict) -> dict:
    """Full inference pipeline — mirrors predictor.py in FastAPI."""
    X_input  = np.array([[features[f] for f in loaded_features]])
    X_scaled = loaded_scaler.transform(X_input)

    predicted_score   = float(np.clip(loaded_regressor.predict(X_scaled)[0], 0, 100))
    failure_prob      = derive_failure_probability(predicted_score)
    predicted_grade   = derive_grade(predicted_score)
    risk_level        = derive_risk_level(failure_prob)

    return {
        'predicted_score'    : round(predicted_score, 2),
        'predicted_grade'    : predicted_grade,
        'failure_probability': failure_prob,
        'risk_level'         : risk_level,
    }

print('Inference functions defined. ✅')

Inference functions defined. ✅


In [15]:
# ── Test Students ─────────────────────────────────────────────────────────────
test_students = {
    'Good Performer': {
        'attendance_percentage'   : 88.0,
        'midterm_score'           : 74.0,
        'historical_gpa'          : 3.2,
        'study_hours_per_week'    : 18.0,
        'subject_difficulty_score': 0.65,
        'ca_avg'                  : 72.0,
        'rule_risk_score'         : 0.18,
    },
    'Average Student': {
        'attendance_percentage'   : 78.0,
        'midterm_score'           : 62.0,
        'historical_gpa'          : 2.6,
        'study_hours_per_week'    : 14.0,
        'subject_difficulty_score': 0.60,
        'ca_avg'                  : 61.0,
        'rule_risk_score'         : 0.32,
    },
    'At-Risk Student': {
        'attendance_percentage'   : 68.0,
        'midterm_score'           : 45.0,
        'historical_gpa'          : 1.8,
        'study_hours_per_week'    : 9.0,
        'subject_difficulty_score': 0.72,
        'ca_avg'                  : 48.0,
        'rule_risk_score'         : 0.52,
    },
}

print('── Sanity Check Predictions ──\n')
print(f"{'Student':<20} {'Score':>7} {'Grade':>6} {'Fail Prob':>10} {'Risk':>10}")
print('-' * 58)
for label, features in test_students.items():
    result = predict_student(features)
    print(
        f"{label:<20} "
        f"{result['predicted_score']:>7.2f} "
        f"{result['predicted_grade']:>6} "
        f"{result['failure_probability']:>10.4f} "
        f"{result['risk_level']:>10}"
    )

── Sanity Check Predictions ──

Student                Score  Grade  Fail Prob       Risk
----------------------------------------------------------
Good Performer         73.19     B+     0.0000        LOW
Average Student        61.44      B     0.4280     MEDIUM
At-Risk Student        50.62     C+     0.9689   CRITICAL


## 11. Final Summary

In [16]:
print('╔══════════════════════════════════════════════════════════╗')
print('║              MODEL TRAINING SUMMARY                     ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Training rows          : {len(X_train):,}                        ║')
print(f'║  Test rows              : {len(X_test):,}                         ║')
print(f'║  Features used          : {len(selected_features)}                           ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Model — XGBoost Regressor (grade_predictor.pkl)        ║')
print(f'║    MAE  : {mae:.4f}                                     ║')
print(f'║    RMSE : {rmse:.4f}                                     ║')
print(f'║    R²   : {r2:.4f}                                     ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Grade letter  : derived from predicted_score            ║')
print('║  Pass/Fail     : derived from predicted_score >= 60      ║')
print('║  Fail prob     : derived from distance to boundary 60    ║')
print('║  Risk level    : derived from failure_probability        ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  PKL Files Saved:                                        ║')
print('║    models/grade_predictor.pkl                            ║')
print('║    models/feature_scaler.pkl                             ║')
print('║    models/feature_list.pkl                               ║')
print('╠══════════════════════════════════════════════════════════╣')
print('║  Next step → 07_model_evaluation.ipynb                   ║')
print('╚══════════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════════╗
║              MODEL TRAINING SUMMARY                     ║
╠══════════════════════════════════════════════════════════╣
║  Training rows          : 4,000                        ║
║  Test rows              : 1,000                         ║
║  Features used          : 7                           ║
╠══════════════════════════════════════════════════════════╣
║  Model — XGBoost Regressor (grade_predictor.pkl)        ║
║    MAE  : 2.5951                                     ║
║    RMSE : 3.2475                                     ║
║    R²   : 0.8437                                     ║
╠══════════════════════════════════════════════════════════╣
║  Grade letter  : derived from predicted_score            ║
║  Pass/Fail     : derived from predicted_score >= 60      ║
║  Fail prob     : derived from distance to boundary 60    ║
║  Risk level    : derived from failure_probability        ║
╠═══════════════════════════════════════════